# 05 - Export for ArcGIS & Final Data Preparation

**目标：**
1. 📤 导出所有分析图层为标准化格式（GeoJSON / GPKG）
2. 🗺️ 准备VIIRS栅格数据给ArcGIS
3. 📋 创建ArcGIS工作流清单
4. 📊 生成元数据文档

**这个notebook为MUSA 6320 StoryMap准备所有必需数据**

## 0. 导入库

In [18]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ 库已导入")

✅ 库已导入


## 1. 配置路径

In [19]:
# 路径配置
DATA_DIR = "./finaldata"
PROCESSED_DIR = os.path.join(DATA_DIR, "processed_data")
EXPORT_DIR = os.path.join(DATA_DIR, "arcgis_export")

# 创建导出目录
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"✅ 导出目录: {EXPORT_DIR}")

# 检查必需文件
required_files = [
    "final_analysis_with_predictions.shp",
    "high_risk_areas.shp",
    "predicted_darkening_areas.shp",
    "complaint_gap_areas.shp"
]

print("\n检查必需文件:")
for filename in required_files:
    filepath = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(filepath):
        print(f"  ✅ {filename}")
    else:
        print(f"  ❌ {filename} - 缺失！")

✅ 导出目录: ./finaldata\arcgis_export

检查必需文件:
  ✅ final_analysis_with_predictions.shp
  ❌ high_risk_areas.shp - 缺失！
  ✅ predicted_darkening_areas.shp
  ✅ complaint_gap_areas.shp


## 2. 导出核心图层 📤

将所有重要图层导出为GeoJSON和GPKG格式

In [20]:
print("\n📤 导出核心图层...\n")

# 定义要导出的图层
layers_to_export = {
    "final_analysis_with_predictions": "完整分析数据（所有block groups + 预测）",
    "high_risk_areas": "高风险维护区域（High + Critical）",
    "predicted_darkening_areas": "预测将变暗的区域",
    "complaint_gap_areas": "Complaint Gap区域（暗但报修少）",
    "priority_areas": "优先维护区域",
    "dark_hotspots": "暗区空间热点（LISA LL区域）"
}

exported_count = 0

for layer_name, description in layers_to_export.items():
    input_file = os.path.join(PROCESSED_DIR, f"{layer_name}.shp")
    
    if not os.path.exists(input_file):
        print(f"⚠️  跳过 {layer_name} (文件不存在)")
        continue
    
    # 读取数据
    try:
        gdf = gpd.read_file(input_file)
        
        # 确保使用WGS84投影（ArcGIS Online标准）
        if gdf.crs != "EPSG:4326":
            gdf = gdf.to_crs("EPSG:4326")
        
        # 导出为GeoJSON
        geojson_file = os.path.join(EXPORT_DIR, f"{layer_name}.geojson")
        gdf.to_file(geojson_file, driver='GeoJSON')
        
        # 导出为GPKG（更适合大数据）
        gpkg_file = os.path.join(EXPORT_DIR, f"{layer_name}.gpkg")
        gdf.to_file(gpkg_file, driver='GPKG', layer=layer_name)
        
        file_size_json = os.path.getsize(geojson_file) / 1024  # KB
        file_size_gpkg = os.path.getsize(gpkg_file) / 1024
        
        print(f"✅ {layer_name}")
        print(f"   - {description}")
        print(f"   - Features: {len(gdf)}")
        print(f"   - GeoJSON: {file_size_json:.1f} KB")
        print(f"   - GPKG: {file_size_gpkg:.1f} KB")
        print()
        
        exported_count += 1
        
    except Exception as e:
        print(f"❌ 导出 {layer_name} 失败: {e}\n")

print(f"\n✅ 成功导出 {exported_count} 个图层")


📤 导出核心图层...

✅ final_analysis_with_predictions
   - 完整分析数据（所有block groups + 预测）
   - Features: 1283
   - GeoJSON: 4922.5 KB
   - GPKG: 2036.0 KB

⚠️  跳过 high_risk_areas (文件不存在)
✅ predicted_darkening_areas
   - 预测将变暗的区域
   - Features: 582
   - GeoJSON: 2278.6 KB
   - GPKG: 988.0 KB

✅ complaint_gap_areas
   - Complaint Gap区域（暗但报修少）
   - Features: 62
   - GeoJSON: 191.1 KB
   - GPKG: 172.0 KB

✅ priority_areas
   - 优先维护区域
   - Features: 1250
   - GeoJSON: 4520.3 KB
   - GPKG: 1872.0 KB

⚠️  跳过 dark_hotspots (文件不存在)

✅ 成功导出 4 个图层


## 3. 创建简化版本（用于web展示）🌐

In [21]:
print("\n🌐 创建简化版本（用于Web地图）...\n")

# 加载完整数据
full_data_file = os.path.join(PROCESSED_DIR, "final_analysis_with_predictions.shp")

if os.path.exists(full_data_file):
    gdf_full = gpd.read_file(full_data_file)
    
    # 选择关键列（减小文件大小）
    key_columns = [
        'geometry',
        'GEOID',
        'median_inc',
        'poverty_ra',
        'population',
        'viirs_2024',
        'streetli_5',
        'crime_per_',
        'is_gap',
        'pri_score',
        'pri_level',
        'risk_score',
        'risk_level',
        'predicted_',  # predicted_brightness_2025
        'brightn_1',   # brightness_decline
        'lisa_label'
    ]
    
    # 找到实际存在的列
    available_cols = [col for col in key_columns if col in gdf_full.columns or 
                     any(gdf_full.columns.str.startswith(col))]
    
    # 精确匹配列名
    cols_to_keep = ['geometry']
    for pattern in key_columns:
        matched = [col for col in gdf_full.columns if col.startswith(pattern)]
        cols_to_keep.extend(matched)
    
    cols_to_keep = list(set(cols_to_keep))  # 去重
    
    if 'geometry' in cols_to_keep:
        gdf_simplified = gdf_full[cols_to_keep].copy()
        
        # 简化几何（减少顶点数）
        if gdf_simplified.crs != "EPSG:4326":
            gdf_simplified = gdf_simplified.to_crs("EPSG:4326")
        
        gdf_simplified['geometry'] = gdf_simplified.geometry.simplify(0.0001)
        
        # 导出简化版
        simplified_file = os.path.join(EXPORT_DIR, "philly_lighting_simplified.geojson")
        gdf_simplified.to_file(simplified_file, driver='GeoJSON')
        
        file_size = os.path.getsize(simplified_file) / 1024
        print(f"✅ 简化版本已创建")
        print(f"   - Features: {len(gdf_simplified)}")
        print(f"   - Columns: {len(cols_to_keep)}")
        print(f"   - File size: {file_size:.1f} KB")
        print(f"   - 文件: philly_lighting_simplified.geojson")
    else:
        print("❌ 无法创建简化版本（缺少geometry列）")
else:
    print("❌ 完整数据文件不存在")


🌐 创建简化版本（用于Web地图）...

✅ 简化版本已创建
   - Features: 1283
   - Columns: 11
   - File size: 949.9 KB
   - 文件: philly_lighting_simplified.geojson


## 4. 准备VIIRS栅格数据 🗺️

In [22]:
print("\n🗺️ 检查VIIRS栅格数据...\n")

# 查找VIIRS栅格文件
viirs_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(('.tif', '.tiff')) and 'viirs' in file.lower():
            viirs_files.append(os.path.join(root, file))

if viirs_files:
    print(f"找到 {len(viirs_files)} 个VIIRS栅格文件:")
    for vf in viirs_files[:5]:  # 显示前5个
        size_mb = os.path.getsize(vf) / (1024*1024)
        print(f"  - {os.path.basename(vf)} ({size_mb:.2f} MB)")
    
    if len(viirs_files) > 5:
        print(f"  ... 和 {len(viirs_files)-5} 个其他文件")
    
    print("\n✅ VIIRS栅格文件已准备好")
    print("   在ArcGIS中导入这些.tif文件进行栅格分析")
else:
    print("⚠️  未找到VIIRS栅格文件")
    print("   请确保01_VIIRS_Data_Extraction已运行并生成了.tif文件")


🗺️ 检查VIIRS栅格数据...

⚠️  未找到VIIRS栅格文件
   请确保01_VIIRS_Data_Extraction已运行并生成了.tif文件


## 5. 生成ArcGIS工作流清单 📋

In [23]:
print("\n📋 生成ArcGIS工作流清单...\n")

workflow_file = os.path.join(EXPORT_DIR, "ArcGIS_Workflow_Checklist.md")

with open(workflow_file, 'w', encoding='utf-8') as f:
    f.write("# ArcGIS Pro 工作流清单 (MUSA 6320 Final)\n\n")
    
    f.write("## 📂 数据导入\n\n")
    f.write("### 矢量数据（从 arcgis_export/ 文件夹导入）\n")
    f.write("- [ ] final_analysis_with_predictions.gpkg - 完整分析数据\n")
    f.write("- [ ] high_risk_areas.gpkg - 高风险区域\n")
    f.write("- [ ] predicted_darkening_areas.gpkg - 预测变暗区域\n")
    f.write("- [ ] complaint_gap_areas.gpkg - Complaint Gap区域\n\n")
    
    f.write("### 栅格数据（VIIRS夜光数据）\n")
    f.write("- [ ] 导入VIIRS .tif文件到ArcGIS Pro\n")
    f.write("- [ ] 确保所有栅格使用相同的投影\n\n")
    
    f.write("---\n\n")
    
    f.write("## 🗺️ 栅格分析工作流 (对应Class 02-06)\n\n")
    
    f.write("### Step 1: Reclassify (Local Processing)\n")
    f.write("- [ ] 完成Reclassify\n")
    f.write("- [ ] 截图保存\n\n")
    
    f.write("### Step 2: Zonal Statistics\n")
    f.write("- [ ] 完成Zonal Statistics\n")
    f.write("- [ ] Join结果到block group图层\n\n")
    
    f.write("### Step 3: Focal Statistics\n")
    f.write("- [ ] 完成Focal Statistics\n")
    f.write("- [ ] 识别低值聚类区域\n\n")
    
    f.write("---\n\n")
    
    f.write("## 🎨 影像分析工作流\n\n")
    f.write("### Unsupervised Classification (推荐)\n")
    f.write("- [ ] 完成分类\n")
    f.write("- [ ] 创建分类地图\n\n")
    
    f.write("---\n\n")
    
    f.write("## 🚶 Cost Surface + Least-Cost Path (必做！)\n\n")
    f.write("### Step 1: 创建Cost Surface\n")
    f.write("- [ ] 创建Cost Surface\n\n")
    
    f.write("### Step 2: Distance Accumulation\n")
    f.write("- [ ] 完成Distance Accumulation\n\n")
    
    f.write("### Step 3: Optimal Path\n")
    f.write("- [ ] 生成Safe Night Routes\n")
    f.write("- [ ] 对比最短路径 vs 最安全路径\n\n")
    
    f.write("---\n\n")
    
    f.write("## 🌐 ArcGIS Online & StoryMap\n\n")
    f.write("- [ ] 发布图层到ArcGIS Online\n")
    f.write("- [ ] 创建Web Map\n")
    f.write("- [ ] 构建StoryMap\n\n")
    
    f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"✅ ArcGIS工作流清单已生成: {workflow_file}")


📋 生成ArcGIS工作流清单...

✅ ArcGIS工作流清单已生成: ./finaldata\arcgis_export\ArcGIS_Workflow_Checklist.md


## 6. 生成数据字典 📖

In [24]:
print("\n📖 生成数据字典...\n")

# 加载主数据以获取字段信息
main_data_file = os.path.join(PROCESSED_DIR, "final_analysis_with_predictions.shp")

if os.path.exists(main_data_file):
    gdf_main = gpd.read_file(main_data_file)
    
    # 生成字段说明
    field_descriptions = {
        'GEOID': '人口普查Block Group ID',
        'median_inc': '中位数家庭收入（美元）',
        'poverty_ra': '贫困率（%）',
        'population': '人口数',
        'no_vehicle': '无车家庭比例（%）',
        'viirs_2024': '2024年VIIRS夜光亮度均值',
        'viirs_all': '所有月份VIIRS亮度均值',
        'viirs_chan': 'VIIRS亮度变化（最新-最早）',
        'viirs_pct': 'VIIRS亮度变化百分比',
        'streetli_5': '311路灯报修数量',
        'crime_per': '犯罪率（每千人）',
        'is_gap': 'Complaint Gap标记（True=Gap区域）',
        'pri_score': '优先级评分（0-100）',
        'pri_level': '优先级等级（Low/Medium/High/Critical）',
        'brightn_1': '预测亮度下降值',
        'predicted': '预测2025年亮度',
        'brightn_2': '亮度趋势斜率',
        'brightn_3': '亮度变化百分比',
        'trend_cat': '趋势类别',
        'risk_score': '综合风险评分（0-100）',
        'risk_level': '风险等级（Low/Medium/High/Critical）',
        'lisa_clust': 'LISA聚类类型（0-4）',
        'lisa_label': 'LISA聚类标签（HH/LL/HL/LH）',
        'is_dark_ho': '是否为暗区热点（LL区域）'
    }
    
    # 保存数据字典
    dict_file = os.path.join(EXPORT_DIR, "Data_Dictionary.md")
    
    with open(dict_file, 'w', encoding='utf-8') as f:
        f.write("# Data Dictionary - Philadelphia Lighting Equity Analysis\n\n")
        f.write(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("## 字段说明\n\n")
        f.write("| 字段名 | 类型 | 说明 |\n")
        f.write("|--------|------|------|\n")
        
        for col in gdf_main.columns:
            if col == 'geometry':
                continue
            
            dtype = str(gdf_main[col].dtype)
            
            # 查找匹配的描述
            description = "(需要补充说明)"
            for key, desc in field_descriptions.items():
                if key in col:
                    description = desc
                    break
            
            f.write(f"| `{col}` | {dtype} | {description} |\n")
        
        f.write("\n## 数据来源\n\n")
        f.write("1. **VIIRS夜光数据**: NASA Black Marble (2022-2024)\n")
        f.write("2. **311报修数据**: OpenDataPhilly\n")
        f.write("3. **犯罪数据**: Philadelphia Police Department\n")
        f.write("4. **人口普查数据**: US Census ACS 5-year estimates\n\n")
        
        f.write("## 投影信息\n\n")
        f.write(f"- **原始CRS**: {gdf_main.crs}\n")
        f.write("- **导出CRS**: EPSG:4326 (WGS84)\n")
        f.write("- **推荐ArcGIS CRS**: EPSG:3857 (Web Mercator) 或 EPSG:2272 (PA State Plane South)\n\n")
        
        f.write("## 关键指标计算方法\n\n")
        f.write("### Complaint Gap\n")
        f.write("定义为同时满足以下条件的区域:\n")
        f.write("- VIIRS亮度 < 33rd百分位数（最暗的1/3）\n")
        f.write("- 311报修率 < 33rd百分位数（报修最少的1/3）\n")
        f.write("- 中位数收入 < 50th百分位数（低于中位数）\n\n")
        
        f.write("### Risk Score\n")
        f.write("加权综合评分 (0-100):\n")
        f.write("- 当前暗度: 30%\n")
        f.write("- 报修不足: 20%\n")
        f.write("- 社会经济脆弱性: 20%\n")
        f.write("- 预测变暗: 15%\n")
        f.write("- 空间聚类: 15%\n\n")
        
        f.write("### LISA Cluster\n")
        f.write("局部Moran's I空间自相关分析:\n")
        f.write("- HH (High-High): 高亮度区域被高亮度邻居包围\n")
        f.write("- LL (Low-Low): 低亮度区域被低亮度邻居包围（暗区热点）\n")
        f.write("- HL (High-Low): 高亮度区域被低亮度邻居包围\n")
        f.write("- LH (Low-High): 低亮度区域被高亮度邻居包围\n")
    
    print(f"✅ 数据字典已生成: {dict_file}")
else:
    print("❌ 主数据文件不存在，无法生成数据字典")


📖 生成数据字典...

✅ 数据字典已生成: ./finaldata\arcgis_export\Data_Dictionary.md


## 7. 生成导出清单总结

In [25]:
print("\n" + "="*60)
print("📦 导出总结")
print("="*60 + "\n")

# 统计导出文件
geojson_files = [f for f in os.listdir(EXPORT_DIR) if f.endswith('.geojson')]
gpkg_files = [f for f in os.listdir(EXPORT_DIR) if f.endswith('.gpkg')]
doc_files = [f for f in os.listdir(EXPORT_DIR) if f.endswith('.md')]

print(f"✅ GeoJSON文件: {len(geojson_files)} 个")
for f in geojson_files:
    print(f"   - {f}")

print(f"\n✅ GPKG文件: {len(gpkg_files)} 个")
for f in gpkg_files:
    print(f"   - {f}")

print(f"\n✅ 文档文件: {len(doc_files)} 个")
for f in doc_files:
    print(f"   - {f}")

# 计算总文件大小
total_size = sum(os.path.getsize(os.path.join(EXPORT_DIR, f)) 
                for f in os.listdir(EXPORT_DIR)) / (1024*1024)

print(f"\n📊 导出文件夹总大小: {total_size:.2f} MB")
print(f"📂 导出位置: {EXPORT_DIR}")

print("\n" + "="*60)
print("🎉 所有数据已准备完毕！")
print("="*60)
print("\n下一步:")
print("1. 在ArcGIS Pro中打开新项目")
print("2. 按照 ArcGIS_Workflow_Checklist.md 执行工作流")
print("3. 完成所有栅格分析和Cost Path分析")
print("4. 发布图层到ArcGIS Online")
print("5. 构建StoryMap")
print("\n⏰ 记住: 6320 deadline是 Dec 5 (比5500早！)")


📦 导出总结

✅ GeoJSON文件: 5 个
   - complaint_gap_areas.geojson
   - final_analysis_with_predictions.geojson
   - philly_lighting_simplified.geojson
   - predicted_darkening_areas.geojson
   - priority_areas.geojson

✅ GPKG文件: 4 个
   - complaint_gap_areas.gpkg
   - final_analysis_with_predictions.gpkg
   - predicted_darkening_areas.gpkg
   - priority_areas.gpkg

✅ 文档文件: 2 个
   - ArcGIS_Workflow_Checklist.md
   - Data_Dictionary.md

📊 导出文件夹总大小: 17.52 MB
📂 导出位置: ./finaldata\arcgis_export

🎉 所有数据已准备完毕！

下一步:
1. 在ArcGIS Pro中打开新项目
2. 按照 ArcGIS_Workflow_Checklist.md 执行工作流
3. 完成所有栅格分析和Cost Path分析
4. 发布图层到ArcGIS Online
5. 构建StoryMap

⏰ 记住: 6320 deadline是 Dec 5 (比5500早！)
